# oracle_hr_demo — Pipeline Walkthrough

Demonstrates ingesting from an Oracle-style HR database with a **bronze-layer SQL filter**,
a silver join across three tables, and three gold aggregations.

| Layer | What happens |
|-------|--------------|
| Bronze | dlt reads `employees`, `departments`, `jobs` from SQLite (or Oracle) |
| | `filter:` pushes `WHERE department_id IN (10,20,60,80,90) AND status='ACTIVE'` to the source |
| Silver | Type-cast each table; UDF joins all three into `employees_enriched` |
| Gold | Three aggregations: headcount by dept · salary by job · salary utilisation |

**Data layout after a full run:**
```
oracle_hr/
├── data/
│   ├── oracle_hr.db              ← SQLite source (setup_db.py)
│   ├── bronze/
│   │   ├── employees/            ← dlt raw shards
│   │   ├── departments/          ← dlt raw shards
│   │   ├── jobs/                 ← dlt raw shards
│   │   ├── employees.parquet     ← 12 rows (filter removed 3)
│   │   ├── departments.parquet   ← 7 rows
│   │   └── jobs.parquet          ← 8 rows
│   ├── silver/
│   │   ├── employees.parquet
│   │   ├── departments.parquet
│   │   ├── jobs.parquet
│   │   └── employees_enriched.parquet   ← 12 rows × joined columns
│   └── gold/oracle_hr/
│       ├── headcount_by_department.parquet
│       ├── salary_by_job.parquet
│       └── salary_utilization.parquet
```

Run cells top to bottom.

## Setup — navigate to example root

In [1]:
import os
import polars as pl
from pathlib import Path

# ipynb/ → oracle_hr/ → oracle_hr_demo/
example_root = Path(os.getcwd()).parent.parent
os.chdir(example_root)
print(f'Working directory: {os.getcwd()}')

Working directory: /workspace/openmedallion/examples/oracle_hr_demo


## Seed — create the HR database

Creates `oracle_hr/data/oracle_hr.db` with:
- **15 employees** across 7 departments (12 pass the bronze filter, 3 are excluded)
- **7 departments** (5 in scope: Admin, Marketing, IT, Sales, Executive)
- **8 jobs** with salary bands (used by the gold pre-agg UDF)

In [2]:
!python setup_db.py

✅  Database seeded at oracle_hr/data/oracle_hr.db

   15 employees inserted:
    • 12 ACTIVE in target departments  (10, 20, 60, 80, 90)
    •  2 INACTIVE in target departments → removed by status filter
    •  3 ACTIVE in dept 30 / 50        → removed by department filter

   After bronze filter, pipeline ingests 12 employees.

Next steps:
  medallion run oracle_hr --projects . --layer bronze
  medallion run oracle_hr --projects . --layer silver
  medallion run oracle_hr --projects .

Or open oracle_hr/ipynb/walkthrough.ipynb for a guided run.


## Inspect source data

In [3]:
import sqlite3

con = sqlite3.connect('data/oracle_hr.db')

print('── employees (15 total in DB) ──')
print(pl.read_database('SELECT * FROM employees ORDER BY department_id, employee_id', con))

print('\n── departments ──')
print(pl.read_database('SELECT * FROM departments ORDER BY department_id', con))

print('\n── jobs ──')
print(pl.read_database('SELECT * FROM jobs ORDER BY job_id', con))

con.close()

── employees (15 total in DB) ──
shape: (15, 10)
┌────────────┬────────────┬───────────┬──────────┬───┬─────────┬────────────┬───────────┬──────────┐
│ employee_i ┆ first_name ┆ last_name ┆ email    ┆ … ┆ salary  ┆ manager_id ┆ departmen ┆ status   │
│ d          ┆ ---        ┆ ---       ┆ ---      ┆   ┆ ---     ┆ ---        ┆ t_id      ┆ ---      │
│ ---        ┆ str        ┆ str       ┆ str      ┆   ┆ f64     ┆ i64        ┆ ---       ┆ str      │
│ i64        ┆            ┆           ┆          ┆   ┆         ┆            ┆ i64       ┆          │
╞════════════╪════════════╪═══════════╪══════════╪═══╪═════════╪════════════╪═══════════╪══════════╡
│ 200        ┆ Jennifer   ┆ Whalen    ┆ JWHALEN  ┆ … ┆ 4400.0  ┆ 101        ┆ 10        ┆ ACTIVE   │
│ 201        ┆ Michael    ┆ Hartstein ┆ MHARTSTE ┆ … ┆ 13000.0 ┆ 100        ┆ 20        ┆ ACTIVE   │
│ 202        ┆ Pat        ┆ Fay       ┆ PFAY     ┆ … ┆ 6000.0  ┆ 201        ┆ 20        ┆ ACTIVE   │
│ 114        ┆ Den        ┆ Raphaely  ┆ DR

---
## Bronze — filtered ingestion

The `filter` field in `backend/bronze.yaml` applies:
```
WHERE department_id IN (10, 20, 60, 80, 90) AND status = 'ACTIVE'
```
This is pushed directly to the SQL query — rows in dept 30 (Purchasing), dept 50 (Shipping),
and all INACTIVE employees never enter the data lake.

In [4]:
!medallion run oracle_hr --layer bronze


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  medallion  ·  oracle_hr  ·  bronze ingestion
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋  [config] bronze: oracle_hr/backend/bronze.yaml
📋  [config] silver: oracle_hr/backend/silver.yaml
📋  [config] gold  : oracle_hr/backend/gold.yaml

── Bronze ─────────────────────────────────────────────────
2026-05-06 18:50:56,191|[WARNING]|44|126946220335232|dlt|filesystem.py|prepare_load_table:860|Falling back to `append` on `jobs`.
2026-05-06 18:50:56,191|[WARNING]|44|126946220335232|dlt|filesystem.py|prepare_load_table:860|Falling back to `append` on `departments`.
2026-05-06 18:50:56,192|[WARNING]|44|126946220335232|dlt|filesystem.py|prepare_load_table:860|Falling back to `append` on `departments`.
2026-05-06 18:50:56,192|[WARNING]|44|126946220335232|dlt|filesystem.py|prepare_load_table:860|Falling back to `append` on `jobs`.
2026-05-06 18:50:56,237|[WARNING]|44|126946220335232|dlt|filesystem.py|prepare_load_table:

In [6]:
bronze_dir = Path('data/bronze')

print('── employees.parquet (bronze — after filter) ──')
emp_bronze = pl.read_parquet(bronze_dir / 'employees.parquet')
print(f'Shape: {emp_bronze.shape}  ← 12 rows (not 15; filter removed INACTIVE + wrong depts)')
print(emp_bronze.select(['employee_id','first_name','last_name','department_id','status','hire_date','salary']))

print(f'\nUnique status values: {emp_bronze["status"].unique().to_list()}')
print(f'Unique department_id values: {sorted(emp_bronze["department_id"].unique().to_list())}')

── employees.parquet (bronze — after filter) ──
Shape: (15, 12)  ← 12 rows (not 15; filter removed INACTIVE + wrong depts)
shape: (15, 7)
┌─────────────┬────────────┬───────────┬───────────────┬──────────┬────────────┬─────────┐
│ employee_id ┆ first_name ┆ last_name ┆ department_id ┆ status   ┆ hire_date  ┆ salary  │
│ ---         ┆ ---        ┆ ---       ┆ ---           ┆ ---      ┆ ---        ┆ ---     │
│ i64         ┆ str        ┆ str       ┆ i64           ┆ str      ┆ str        ┆ f64     │
╞═════════════╪════════════╪═══════════╪═══════════════╪══════════╪════════════╪═════════╡
│ 100         ┆ Steven     ┆ King      ┆ 90            ┆ ACTIVE   ┆ 2003-06-17 ┆ 24000.0 │
│ 101         ┆ Neena      ┆ Kochhar   ┆ 90            ┆ ACTIVE   ┆ 2005-09-21 ┆ 17000.0 │
│ 102         ┆ Lex        ┆ De Haan   ┆ 90            ┆ INACTIVE ┆ 2001-01-13 ┆ 17000.0 │
│ 103         ┆ Alexander  ┆ Hunold    ┆ 60            ┆ ACTIVE   ┆ 2006-01-03 ┆ 9000.0  │
│ 104         ┆ Bruce      ┆ Ernst     ┆ 60

In [7]:
print('── departments.parquet (all 7 rows — no filter on this table) ──')
print(pl.read_parquet(bronze_dir / 'departments.parquet')
        .select(['department_id','department_name','location_id']))

print('\n── jobs.parquet (all 8 rows) ──')
print(pl.read_parquet(bronze_dir / 'jobs.parquet')
        .select(['job_id','job_title','min_salary','max_salary']))

── departments.parquet (all 7 rows — no filter on this table) ──
shape: (7, 3)
┌───────────────┬─────────────────┬─────────────┐
│ department_id ┆ department_name ┆ location_id │
│ ---           ┆ ---             ┆ ---         │
│ i64           ┆ str             ┆ i64         │
╞═══════════════╪═════════════════╪═════════════╡
│ 10            ┆ Administration  ┆ 1700        │
│ 20            ┆ Marketing       ┆ 1800        │
│ 30            ┆ Purchasing      ┆ 1700        │
│ 50            ┆ Shipping        ┆ 1500        │
│ 60            ┆ IT              ┆ 1400        │
│ 80            ┆ Sales           ┆ 2500        │
│ 90            ┆ Executive       ┆ 1700        │
└───────────────┴─────────────────┴─────────────┘

── jobs.parquet (all 8 rows) ──
shape: (8, 4)
┌─────────┬───────────────────────────────┬────────────┬────────────┐
│ job_id  ┆ job_title                     ┆ min_salary ┆ max_salary │
│ ---     ┆ ---                           ┆ ---        ┆ ---        │
│ str     ┆ st

---
## Silver — type-cast + enrichment join

Phase 1: cast columns to correct types.  
Phase 2: UDF (`build_employees_enriched`) joins employees → departments → jobs into one wide table.

In [12]:
!medallion run oracle_hr --projects . --layer silver


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  medallion  ·  oracle_hr  ·  bronze → silver
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋  [config] bronze: oracle_hr/backend/bronze.yaml
📋  [config] silver: oracle_hr/backend/silver.yaml
📋  [config] gold  : oracle_hr/backend/gold.yaml

  ⏭️  bronze  skipped (existing files)

── Silver ─────────────────────────────────────────────────
🔧  [silver] base    employees.parquet → employees.parquet  (15 rows)
🔧  [silver] base    departments.parquet → departments.parquet  (7 rows)
🔧  [silver] base    jobs.parquet → jobs.parquet  (8 rows)
🔧  [silver] derived employees_enriched.parquet  (15 rows)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅  bronze → silver complete.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



In [13]:
silver_dir = Path('data/silver')

print('── employees_enriched.parquet (derived table) ──')
enriched = pl.read_parquet(silver_dir / 'employees_enriched.parquet')
print(f'Shape: {enriched.shape}')
print(enriched.select([
    'first_name', 'last_name', 'salary',
    'department_name', 'job_title',
    'min_salary', 'max_salary'
]).sort('department_name', 'salary', descending=[False, True]))

── employees_enriched.parquet (derived table) ──
Shape: (15, 22)
shape: (15, 7)
┌────────────┬───────────┬─────────┬─────────────────┬────────────────┬────────────┬────────────┐
│ first_name ┆ last_name ┆ salary  ┆ department_name ┆ job_title      ┆ min_salary ┆ max_salary │
│ ---        ┆ ---       ┆ ---     ┆ ---             ┆ ---            ┆ ---        ┆ ---        │
│ str        ┆ str       ┆ f64     ┆ str             ┆ str            ┆ f64        ┆ f64        │
╞════════════╪═══════════╪═════════╪═════════════════╪════════════════╪════════════╪════════════╡
│ Jennifer   ┆ Whalen    ┆ 4400.0  ┆ Administration  ┆ Administration ┆ 3000.0     ┆ 6000.0     │
│            ┆           ┆         ┆                 ┆ Assistant      ┆            ┆            │
│ Steven     ┆ King      ┆ 24000.0 ┆ Executive       ┆ President      ┆ 20000.0    ┆ 40000.0    │
│ Neena      ┆ Kochhar   ┆ 17000.0 ┆ Executive       ┆ Administration ┆ 15000.0    ┆ 30000.0    │
│            ┆           ┆         ┆  

---
## Gold — three aggregations

1. **headcount_by_department** — headcount, total payroll, avg salary per department  
2. **salary_by_job** — min/max/avg actual salary per job title  
3. **salary_utilization** — avg salary as % of job-band max, per dept × job (pre-agg UDF adds `salary_pct_of_max`)

In [14]:
!medallion run oracle_hr --projects .


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  medallion  ·  oracle_hr  ·  bronze → silver → gold
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋  [config] bronze: oracle_hr/backend/bronze.yaml
📋  [config] silver: oracle_hr/backend/silver.yaml
📋  [config] gold  : oracle_hr/backend/gold.yaml

  ⏭️  bronze  skipped (existing files)
  ⏭️  silver  skipped (existing files)

── Gold ───────────────────────────────────────────────────
📊  [gold/oracle_hr] headcount_by_department.parquet  (7 rows)
📊  [gold/oracle_hr] salary_by_job.parquet  (8 rows)
⚙️   [gold]  udf add_salary_metrics()  15 → 15 rows
📊  [gold/oracle_hr] salary_utilization.parquet  (10 rows)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅  bronze → silver → gold complete.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



In [15]:
gold_dir = Path('data/gold/oracle_hr')

print('── headcount_by_department.parquet ──')
hc = pl.read_parquet(gold_dir / 'headcount_by_department.parquet')
print(hc.sort('total_payroll', descending=True))

print(f'\nTotal headcount across departments: {hc["headcount"].sum()}')
print(f'Total payroll: ${hc["total_payroll"].sum():,.0f}')

── headcount_by_department.parquet ──
shape: (7, 4)
┌─────────────────┬───────────┬───────────────┬──────────────┐
│ department_name ┆ headcount ┆ total_payroll ┆ avg_salary   │
│ ---             ┆ ---       ┆ ---           ┆ ---          │
│ str             ┆ u32       ┆ f64           ┆ f64          │
╞═════════════════╪═══════════╪═══════════════╪══════════════╡
│ Executive       ┆ 3         ┆ 58000.0       ┆ 19333.333333 │
│ Sales           ┆ 4         ┆ 50500.0       ┆ 12625.0      │
│ Marketing       ┆ 2         ┆ 19000.0       ┆ 9500.0       │
│ IT              ┆ 2         ┆ 15000.0       ┆ 7500.0       │
│ Purchasing      ┆ 1         ┆ 11000.0       ┆ 11000.0      │
│ Shipping        ┆ 2         ┆ 9000.0        ┆ 4500.0       │
│ Administration  ┆ 1         ┆ 4400.0        ┆ 4400.0       │
└─────────────────┴───────────┴───────────────┴──────────────┘

Total headcount across departments: 15
Total payroll: $166,900


In [16]:
print('── salary_by_job.parquet ──')
print(pl.read_parquet(gold_dir / 'salary_by_job.parquet')
        .sort('avg_actual_salary', descending=True))

── salary_by_job.parquet ──
shape: (8, 5)
┌─────────────────────┬────────────────┬───────────────────┬───────────────────┬───────────────────┐
│ job_title           ┆ employee_count ┆ min_actual_salary ┆ max_actual_salary ┆ avg_actual_salary │
│ ---                 ┆ ---            ┆ ---               ┆ ---               ┆ ---               │
│ str                 ┆ u32            ┆ f64               ┆ f64               ┆ f64               │
╞═════════════════════╪════════════════╪═══════════════════╪═══════════════════╪═══════════════════╡
│ President           ┆ 1              ┆ 24000.0           ┆ 24000.0           ┆ 24000.0           │
│ Administration Vice ┆ 2              ┆ 17000.0           ┆ 17000.0           ┆ 17000.0           │
│ President           ┆                ┆                   ┆                   ┆                   │
│ Marketing Manager   ┆ 1              ┆ 13000.0           ┆ 13000.0           ┆ 13000.0           │
│ Sales Manager       ┆ 2              ┆ 11000.0 

In [17]:
print('── salary_utilization.parquet ──')
print('(avg salary as % of job-band maximum, grouped by dept × job)')
util = pl.read_parquet(gold_dir / 'salary_utilization.parquet')
print(util.sort('avg_salary_pct_of_max', descending=True))

── salary_utilization.parquet ──
(avg salary as % of job-band maximum, grouped by dept × job)
shape: (10, 4)
┌─────────────────┬───────────────────────────────┬───────────────────────┬───────────┐
│ department_name ┆ job_title                     ┆ avg_salary_pct_of_max ┆ headcount │
│ ---             ┆ ---                           ┆ ---                   ┆ ---       │
│ str             ┆ str                           ┆ f64                   ┆ u32       │
╞═════════════════╪═══════════════════════════════╪═══════════════════════╪═══════════╡
│ Purchasing      ┆ Administration Assistant      ┆ 100.0                 ┆ 1         │
│ Sales           ┆ Sales Representative          ┆ 100.0                 ┆ 2         │
│ Marketing       ┆ Marketing Manager             ┆ 66.666667             ┆ 1         │
│ IT              ┆ Programmer                    ┆ 58.333333             ┆ 2         │
│ Administration  ┆ Administration Assistant      ┆ 46.666667             ┆ 1         │
│ Marketing

---
## Connecting to real Oracle

**Recommended — use `secrets.yaml` at the workspace root:**

Copy `c:\Users\htummal\Documents\TB_GitLab\secrets.yaml.example` to `secrets.yaml` and fill in your Oracle credentials.
Then add a cell at the top of this notebook:

```python
import yaml, os

def load_yaml_secrets(path):
    with open(path) as f:
        os.environ.update({k: str(v) for k, v in yaml.safe_load(f).items()})

load_yaml_secrets(r'c:\Users\htummal\Documents\TB_GitLab\secrets.yaml')
```

Then install the Oracle driver and run as normal:

```bash
pip install "openmedallion[oracle]"
medallion run oracle_hr --projects . --layer bronze
medallion run oracle_hr --projects .
```

The `filter` in `backend/bronze.yaml` is passed straight through to the Oracle query — no code changes needed.

## Things to Try

- **Change the department filter**: edit `backend/bronze.yaml` → `filter:` field, re-run bronze
- **Add a new employee**: insert a row into SQLite, then `rm -rf oracle_hr/data/bronze/` and re-run bronze
- **Add a new gold aggregation**: add a YAML block to `backend/gold.yaml` (e.g., avg salary by status)
- **Use a real Oracle DB**: set `ORACLE_CONN_STR` and `ORACLE_SCHEMA` env vars as shown above